# Covariate PCs: fit on the unrelated, project everyone else (nobody dropped)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run, repo_root
from aoucov.batch import run_script
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

In [ ]:
from stage_helpers import hexbin_safe
STAGE = "covariate_pcs"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
KEEP = single_target(cfg, "ancestry_round2")

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT, tools={"bigsnpr": "bed_autoSVD + bed_projectSelfPCA"}):
        run_script(cfg, "covariate_pcs.R", dict(
            BED=single_target(cfg, "hm3_panel"), KEEP=KEEP, RELATEDNESS=cfg.input("relatedness"),
            KIN_CUT=cfg.param("cov_kin_cut"), K=cfg.param("cov_n_pcs"), THR_R2=cfg.param("cov_r2"),
            LD_REGIONS=cfg.input("ld_regions"), OUT=OUT, NCORES=THREADS))

## Checks

In [ ]:
pcs = pd.read_csv(OUT, sep="\t", dtype={"person_id": str})
meta = json.load(open(OUT.replace(".tsv", "_meta.json")))
final = pd.read_csv(KEEP, header=None, dtype=str)[0]
rel = pd.read_csv(cfg.input("relatedness"), sep="\t", dtype={"i.s": str, "j.s": str})
rel = rel[(rel["kin"] > cfg.param("cov_kin_cut")) & rel["i.s"].isin(final) & rel["j.s"].isin(final)]
fit_ids = set(pcs.loc[pcs["in_fit"], "person_id"])

rep.count("n_final", len(final)); rep.count("n_fit", meta["n_fit"]); rep.count("n_projected", meta["n_projected"])
rep.note(n_variants_used=meta["n_variants_used"], n_long_range_ld_regions=meta["n_lrldr_regions"])
rep.check("every participant in the final set has PCs", set(pcs["person_id"]) == set(final))
rep.check("no related pair inside the fit set",
          not (rel["i.s"].isin(fit_ids) & rel["j.s"].isin(fit_ids)).any())

In [ ]:
sv = np.array(meta["singular_values"])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(range(1, len(sv) + 1), sv ** 2 / (sv ** 2).sum()); axes[0].set(xlabel="PC", ylabel="share of top-K variance")
for ax, fit in zip(axes[1:], [True, False]):
    d = pcs[pcs["in_fit"] == fit]
    hexbin_safe(ax, d["PC1"], d["PC2"], gridsize=60)
    ax.set(title="fit" if fit else "projected (has a relative)", xlabel="PC1", ylabel="PC2")
rep.fig(fig, "covariate_pcs"); plt.show()
rep.save()